In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm.notebook import tqdm
import random
import numpy as np
from PIL import Image
import json

import torch
import torch.nn as nn
from torchvision import models
from torchsummary import summary
from torchvision import datasets, transforms
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Mission 1-1

In [ ]:
# 파일명에서 특징을 추출하여 데이터프레임으로 정리하는 함수
def extract_features_from_directory(directory):
    data = {
        "File_Name": [],
        "W/T": [],
        "ImgaeID": [],
        "Era": [],
        "Style": [],
        "Gender": []
    }
    
    # 디렉토리 내 파일 탐색
    for filename in os.listdir(directory):
        if filename.endswith(".jpg"):  # 확장자가 jpg인 파일만 처리
            data["File_Name"].append(filename)
            parts = filename.replace(".jpg", "").split("_")
            
            data["W/T"].append(parts[0])
            data["ImgaeID"].append(parts[1])
            data["Era"].append('20'+parts[2] if int(parts[2])<50 else '19'+parts[2])
            data["Style"].append(parts[3])
            data["Gender"].append("여성" if parts[4] == "W" else "남성")
    
    return pd.DataFrame(data)

#### 1-1.  주어진 이미지 데이터의 파일 명은 아래와 같은 형식이다. “{W/T}_{이미지ID}_{시대별}_{스타일별}_{성별}.jpg”이에 기반하여 “이미지ID” 수 기준으로 “성별 & 스타일” 통계치를 아래 표 형식으로 기입한다.

## Train Images 분석

In [ ]:
train_image_dir = './data/training_image'
valid_image_dir = './data/validation_image'

In [ ]:
train_df = extract_features_from_directory(train_image_dir)
train_df.describe()

In [ ]:
# 'Gender'와 'Style' 열을 선택하여 새로운 DataFrame 생성
gender_style_train_df = train_df[['Gender', 'Style']]

# 결과 출력
print(gender_style_train_df)

In [ ]:
# 성별(Gender)과 스타일(Style)로 그룹화하여 각 그룹의 개수를 계산
gender_style_counts = gender_style_train_df.groupby(['Gender', 'Style']).size().reset_index(name='count')

# 결과 출력
print(gender_style_counts)

In [ ]:
# 저장할 디렉토리와 파일 경로 설정
output_dir = "./output"
csv_file_path = os.path.join(output_dir, "1_1_train_images.csv")

# 디렉토리가 없으면 생성
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# DataFrame을 csv 파일로 저장
gender_style_counts.to_csv(csv_file_path, index=False)

# 성공 메시지 출력
print('저장 성공!!\n경로 :', csv_file_path)

## Valid Images 분석

In [ ]:
valid_df = extract_features_from_directory(valid_image_dir)
valid_df.describe()

In [ ]:
# 'Gender'와 'Style' 열을 선택하여 새로운 DataFrame 생성
gender_style_valid_df = valid_df[['Gender', 'Style']]

# 결과 출력
print(gender_style_valid_df)

In [ ]:
# 성별(Gender)과 스타일(Style)로 그룹화하여 각 그룹의 개수를 계산
gender_style_counts = gender_style_valid_df.groupby(['Gender', 'Style']).size().reset_index(name='count')

# 결과 출력
print(gender_style_counts)

In [ ]:
# 저장할 디렉토리와 파일 경로 설정
output_dir = "./output"
csv_file_path = os.path.join(output_dir, "1_1_valid_images.csv")

# 디렉토리가 없으면 생성
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# DataFrame을 csv 파일로 저장
gender_style_counts.to_csv(csv_file_path, index=False)

# 성공 메시지 출력
print('저장 성공!!\n경로 :', csv_file_path)

# Mission 1-2

##  1-2.  ResNet-18를 활용하여 “성별 & 스타일” 단위로 클래스 분류를 수행하고 Validation 데이터에 대한 정확도를 제시한다.

### ResNet-18 정의하기
##### https://github.com/pytorch/vision/blob/main/torchvision/models/resnet.py 참고

##### ResNet-18 Sample

In [ ]:
# 시드 고정 함수
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    random.seed(seed)
    np.random.seed(seed)

set_seed()  # 시드 고정

### 모든 Image dataset들을 segmentation하여 새로 저장 (pretrained DeepLab v3 사용)

In [ ]:
# 모델 로드 (DeepLabV3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pre_model = models.segmentation.deeplabv3_resnet101(weights="DEFAULT").to(device)
pre_model.eval()

# 이미지 전처리 함수
def preprocess_image(image_path):
    input_image = Image.open(image_path).convert("RGB")
    preprocess = transforms.Compose([
        transforms.Resize((520, 520)),  # DeeplabV3의 기본 입력 크기
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    input_tensor = preprocess(input_image)
    input_batch = input_tensor.unsqueeze(0)  # 배치 차원 추가
    return input_batch

# Segmentation 함수
def segment_person(image_path):
    input_tensor = preprocess_image(image_path).to(device)
    with torch.no_grad():
        output = pre_model(input_tensor)['out'][0]  # 모델의 출력

    output_predictions = output.argmax(0).cpu().numpy()  # 예측된 클래스 번호

    # 사람에 해당하는 클래스(15)의 마스크 생성
    person_mask = np.where(output_predictions == 15, 1, 0).astype(np.uint8)

    # 여러 사람 영역 분리
    num_labels, labels_im = cv2.connectedComponents(person_mask)

    # 가장 큰 사람 영역 선택
    max_label = 1 + np.argmax([np.sum(labels_im == i) for i in range(1, num_labels)])
    best_person_mask = (labels_im == max_label).astype(np.uint8)

    # 바운딩 박스 계산
    y_indices, x_indices = np.where(best_person_mask == 1)
    min_x, max_x = x_indices.min(), x_indices.max()
    min_y, max_y = y_indices.min(), y_indices.max()
    bbox = (min_x, min_y, max_x, max_y)  # 바운딩 박스 좌표
    
    return best_person_mask, bbox

# 사람 영역을 crop하고 520x520으로 리사이즈
def crop_and_resize_person(image_path, person_mask, bbox, target_size=(520, 520)):
    original_image = Image.open(image_path).convert("RGB").resize((520, 520))
    original_array = np.array(original_image)

    # 바운딩 박스 기준으로 이미지와 마스크를 crop
    min_x, min_y, max_x, max_y = bbox
    cropped_image = original_array[min_y:max_y, min_x:max_x]
    cropped_mask = person_mask[min_y:max_y, min_x:max_x]

    # 사람 영역이 전체 이미지의 1/5 미만인 경우 조건 적용
    mask_area = np.sum(person_mask)
    image_area = person_mask.shape[0] * person_mask.shape[1]
    if mask_area < image_area / 10:
        # 중앙 기준 가로 3/5, 세로 전체로 crop
        center_x = original_array.shape[1] // 2
        width_3_5 = original_array.shape[1] * 3 // 5
        min_x = max(center_x - width_3_5 // 2, 0)
        max_x = min(center_x + width_3_5 // 2, original_array.shape[1])
        cropped_image = original_array[:, min_x:max_x]
        
        # RGBA 포맷으로 변환 (배경 투명)
        transparent_image = np.zeros((cropped_image.shape[0], cropped_image.shape[1], 4), dtype=np.uint8)
        transparent_image[..., :3] = cropped_image
        transparent_image[..., 3] = 255  # 불투명하게 설정
    else:
        # mask 영역만 남기고 나머지 투명하게 처리
        transparent_image = np.zeros((cropped_image.shape[0], cropped_image.shape[1], 4), dtype=np.uint8)
        transparent_image[..., :3] = cropped_image
        transparent_image[..., 3] = cropped_mask * 255  # 사람 영역을 불투명하게

    # 520x520 크기로 리사이즈
    resized_image = Image.fromarray(transparent_image, 'RGBA').resize(target_size, Image.LANCZOS)
    return resized_image

# 디렉토리 내 모든 이미지 처리 후 저장
def process_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    # tqdm을 사용하여 진행 상황을 표시
    for filename in tqdm(os.listdir(input_dir), desc="Processing Images", unit="image"):
        if filename.endswith('.jpg'):
            image_path = os.path.join(input_dir, filename)
            person_mask, bbox = segment_person(image_path)
            resized_image = crop_and_resize_person(image_path, person_mask, bbox, target_size=(520, 520))
            
            # 알파 채널을 제거하여 RGB로 변환
            resized_image = resized_image.convert("RGB")
            
            # 저장 경로 설정
            output_path = os.path.join(output_dir, filename)
            resized_image.save(output_path, format='JPEG')



#### Train image 전처리

In [ ]:
# 실행 코드
input_dir = 'data/training_image'  # 이미지 디렉토리 경로 지정
output_dir = 'data/final_seg_train_image'  # 결과 저장 디렉토리 경로 지정
process_directory(input_dir, output_dir)

#### Valid image 전처리

In [ ]:
# 실행 코드
input_dir = 'data/validation_image'  # 이미지 디렉토리 경로 지정
output_dir = 'data/final_seg_valid_image'  # 결과 저장 디렉토리 경로 지정
process_directory(input_dir, output_dir)

------------------------

In [ ]:
# 딥러닝 하이퍼파라미터를 변수로 정의 (대문자로 변경)

# 학습 파라미터
LEARNING_RATE = 0.05       # 학습률
BATCH_SIZE = 32              # 배치 크기
NUM_EPOCHS = 500             # 에포크 수
OPTIMIZER = 'AdamW'         # 최적화 알고리즘 (Adam, SGD, AdamW 등)
WEIGHT_DECAY = 1e-4         # AdamW와 같은 옵티마이저에서의 가중치 감쇠
MOMENTUM = 0.98              # SGD와 같은 옵티마이저에서의 모멘텀

# 모델 파라미터
NUM_CLASSES = 31            # 분류할 클래스 수

# 데이터셋 파라미터
TRAIN_DATA_DIR = './data/final_seg_train_image/'  # 학습 데이터 경로
VALID_DATA_DIR = './data/final_seg_valid_image'  # 검증 데이터 경로

# 장치 파라미터
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # GPU 또는 CPU 사용


## 분류기 생성

In [ ]:
# 분류할 클래스 수
NUM_CLASSES = 31

# ResNet-18 모델 불러오기 (weights=None로 초기화된 모델 사용)
model = models.resnet18(weights=None)

# He 초기화를 위한 함수 정의
def init_weights_he(model):
    if isinstance(model, nn.Conv2d) or isinstance(model, nn.Linear):
        torch.nn.init.kaiming_normal_(model.weight, mode='fan_out', nonlinearity='leaky_relu')
        if model.bias is not None:
            torch.nn.init.constant_(model.bias, 0)

# 모델의 모든 레이어에 대해 He 초기화 적용
model.apply(init_weights_he)

# 출력 클래스를 원하는 개수로 변경 (예: 31개의 클래스를 분류하는 경우)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# 마지막 fully connected 레이어에 초기화 적용
torch.nn.init.kaiming_normal_(model.fc.weight, mode='fan_out', nonlinearity='leaky_relu')
torch.nn.init.constant_(model.fc.bias, 0)

# GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델을 GPU로 이동
model = model.to(device)

# 모델 구조 확인
summary(model, (3, 384, 384))


### Dataset구축하기

In [ ]:
# Gender와 Style을 결합하여 새로운 Class 열 생성
train_df['Class'] = train_df['Gender'] + '_' + train_df['Style']
valid_df['Class'] = valid_df['Gender'] + '_' + valid_df['Style']

# 각 클래스에 고유한 숫자 인덱스 부여 (Class 열을 카테고리로 변환 후 숫자 할당)
class_mapping = {cls: idx for idx, cls in enumerate(train_df['Class'].unique())}
train_df['Class_Index'] = train_df['Class'].map(class_mapping)
valid_df['Class_Index'] = valid_df['Class'].map(class_mapping)

# File_Name과 인덱스를 결합
train_dataset = train_df[['File_Name', 'Class_Index']]
valid_dataset = valid_df[['File_Name', 'Class_Index']]

# 결과 CSV 파일로 저장
train_csv_file = '1train_dataset.csv'
valid_csv_file = '1valid_dataset.csv'

train_dataset.to_csv(train_csv_file, index=False)
valid_dataset.to_csv(valid_csv_file, index=False)


In [ ]:
class_mapping

In [ ]:
# indexing으로 실제 class name을 찾을 수 있는 list
class_names = list(class_mapping.keys())

#### Final_seg_train_image의 Mean, Std 계산하기

In [ ]:
# 이미지 디렉토리 경로 설정
img_dir = './data/final_seg_train_image/'

# 이미지 전처리 (384*384로 리사이즈하고, Normalize 없이 텐서로 변환)
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor()
])

# RGB 채널별 합을 계산하기 위한 변수 초기화
mean = torch.zeros(3)
std = torch.zeros(3)
num_samples = 0

# 디렉토리 순회하며 이미지 로드
for img_file in os.listdir(img_dir):
    if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(img_dir, img_file)
        
        # 이미지 로드 및 전처리
        image = Image.open(img_path).convert("RGB")
        image = transform(image)  # [C, H, W] 형식의 텐서로 변환
        
        # 전체 픽셀 수 업데이트
        num_samples += image.size(1) * image.size(2)  # Height x Width
        
        # 채널별 합 및 표준 편차 계산
        mean += image.mean([1, 2])  # [1, 2]는 Height와 Width 차원
        std += image.std([1, 2])    # [1, 2]는 Height와 Width 차원

# 평균 및 표준 편차 계산
mean /= len(os.listdir(img_dir))
std /= len(os.listdir(img_dir))

print(f"Mean (RGB) for train images: {mean}")
print(f"Std (RGB) for train images: {std}")


### transform에 적용하세요

_________________________________________________

In [ ]:
# CustomImageDataset 클래스 정의
class CustomImageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data_frame.iloc[idx, 0])
        image = Image.open(img_name).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = int(self.data_frame.iloc[idx, 1])
        return image, label

In [ ]:
# 각 전처리 및 증강 기법 정의 384
train_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.2651, 0.2375, 0.2316], [0.2848, 0.2609, 0.2559])
])

train_HorizontalFlip_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(p=0.8),  # 80% 확률로 수평 반전
    transforms.RandomApply([transforms.RandomRotation(degrees=15)], p=0.2),  # 20% 확률로 ±15도 회전
    transforms.ColorJitter(
        brightness=0.2,  # 밝기 조정 (0.2 범위 내에서 밝기를 랜덤하게 조절)
        hue=0.1          # 색조 조정 (0.1 범위 내에서 색조를 랜덤하게 조절)
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.2651, 0.2375, 0.2316], [0.2848, 0.2609, 0.2559])
])

val_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.2651, 0.2375, 0.2316], [0.2848, 0.2609, 0.2559])
])

# 각 데이터셋 로드
train_dataset = CustomImageDataset(csv_file=train_csv_file, img_dir=TRAIN_DATA_DIR, transform=train_transform)
train_HorizontalFlip_dataset = CustomImageDataset (csv_file=train_csv_file, img_dir=TRAIN_DATA_DIR, transform=train_HorizontalFlip_transform)

val_dataset = CustomImageDataset(csv_file=valid_csv_file, img_dir=VALID_DATA_DIR, transform=val_transform)

# 여러 데이터셋을 하나로 병합
concat_dataset = ConcatDataset([train_dataset, train_HorizontalFlip_dataset]) #, train_ColorJitter_dataset

# DataLoader 생성
train_loader = DataLoader(concat_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
# 손실 함수 및 옵티마이저 설정
criterion = nn.CrossEntropyLoss()  # 분류 작업을 위한 손실 함수 (One-Hot-Encoding이 아니라 클래스 인덱스를 사용하는 경우)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ReduceLROnPlateau 스케줄러 설정 (검증 손실이 2번 개선되지 않으면 학습률 감소)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=4, factor=0.90, verbose=True)

In [ ]:
# Accuracy 기반의 early stopping class
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_accuracy = None
        self.early_stop = False

    def __call__(self, val_accuracy):
        if self.best_accuracy is None:
            self.best_accuracy = val_accuracy
        elif val_accuracy < self.best_accuracy + self.min_delta:  # 정확도가 개선되지 않으면
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_accuracy = val_accuracy
            self.counter = 0


# Early Stopping 인스턴스 생성
early_stopping = EarlyStopping(patience=20, min_delta=0.1) # accuracy 기반에서는 0.1, loss에서는 0.01


In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc='Training', leave=False, total=len(train_loader))

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # 모델 예측
        outputs = model(images)

        # 손실 계산
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 손실 누적
        running_loss += loss.item() * images.size(0)

        # 예측값과 실제값으로 정확도 계산 (argmax 사용)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # tqdm 진행 상황 업데이트
        progress_bar.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    # 에폭이 끝난 후 한 번만 최종 손실 및 정확도 출력
    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    # 최종 결과 로그 출력
    #tqdm.write(f'Final Epoch Loss: {epoch_loss:.4f}, Final Accuracy: {epoch_accuracy:.2f}%')

    #scheduler.step()

    return epoch_loss, epoch_accuracy


In [ ]:
def validate(model, valid_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # tqdm 진행 바 설정
    progress_bar = tqdm(valid_loader, desc='Validation', leave=False)

    with torch.no_grad():
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)

            # 모델 예측 (logits 사용)
            outputs = model(images)

            # 손실 계산
            loss = criterion(outputs, labels)

            # 손실 누적
            running_loss += loss.item() * images.size(0)

            # 예측값과 실제값으로 정확도 계산 (argmax 사용)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # tqdm 진행 상황 업데이트
            progress_bar.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    # 에폭이 끝난 후 한 번만 최종 손실 및 정확도 출력
    val_loss = running_loss / total
    val_accuracy = 100 * correct / total

    # 최종 결과 로그 출력
    #tqdm.write(f'Final Validation Loss: {val_loss:.4f}, Final Accuracy: {val_accuracy:.2f}%')

    return val_loss, val_accuracy


In [ ]:
# Output 디렉토리 생성 (존재하지 않으면 생성)
output_dir = './output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 학습 및 검증 과정을 기록하기 위한 리스트
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

# 최적의 val_loss 초기값 설정
best_val_loss = float('inf')
best_accuracy = 0

# tqdm를 이용한 에포크 진행 표시
epoch_bar = tqdm(range(NUM_EPOCHS), desc="Total Epoch Progress", total=NUM_EPOCHS)

# 전체 학습 루프
for epoch in epoch_bar:
    print(f'Epoch {epoch + 1}/{NUM_EPOCHS}')

    # 한 epoch 동안의 학습 진행
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # 검증 단계 진행
    val_loss, val_acc = validate(model, valid_loader, criterion, device)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    # 결과 출력
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

    # 최적의 val_accuracy를 기록하고 모델 저장 (추가 정보 포함)
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        best_model_path = os.path.join(output_dir, 'best_model.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'best_accuracy': best_accuracy,
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict()
        }, best_model_path)
        print(f'Best model saved to {best_model_path} with val_accuracy: {best_accuracy:.2f}%')
    
    
    # Early Stopping 체크
    # early_stopping(val_loss)
    early_stopping(val_acc)
    if early_stopping.early_stop:
        print("Early stopping triggered!")
        break  # 학습 중지
        
    # 검증 손실을 기반으로 스케줄러 업데이트
    scheduler.step(val_loss) 

# 학습이 끝난 후 최종 모델 저장
final_model_path = os.path.join(output_dir, 'final_model.pth')
torch.save({
    'epoch': epoch + 1,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}, final_model_path)
print(f'Final model saved to {final_model_path}')


In [ ]:
# Loss 및 Accuracy 그래프 출력 및 저장
def plot_and_save_metrics(train_losses, train_accuracies, val_losses, val_accuracies, output_dir):
    epochs = range(1, len(train_losses) + 1)
    
    # Loss 그래프
    plt.figure(figsize=(14, 5))
    
    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.plot(epochs, val_losses, label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Accuracy 그래프
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label='Train Accuracy')
    plt.plot(epochs, val_accuracies, label='Validation Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    # 그래프 파일 저장
    plot_save_path = os.path.join(output_dir, 'training_plot.png')
    plt.savefig(plot_save_path)
    print(f'Plot saved to {plot_save_path}')
    plt.show()

# Loss 및 Accuracy 그래프 출력 및 저장
plot_and_save_metrics(train_losses, train_accuracies, val_losses, val_accuracies, output_dir)

In [ ]:
# 시드 고정
set_seed()

# 모델 준비 및 가중치 로드
NUM_CLASSES = 31
model = models.resnet18(weights=None)

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)


# best_model.pth에서 모든 상태를 로드
checkpoint = torch.load("/home/user/Desktop/ICT/output/best_model.pth", map_location=device)
print('Saved Best Acc :',checkpoint['best_accuracy'])
model.load_state_dict(checkpoint['model_state_dict'])


model = model.to(device)
model.eval()  # 평가 모드 설정

# Validation 수행 및 결과 출력
val_loss, val_accuracy = validate(model, valid_loader, criterion, device)
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")


#### Option. get good/bad answer in plot

In [ ]:
# Custom Dataset 클래스 정의
class CustomImageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_name = self.data_frame.iloc[idx, 0]
        img_path = os.path.join(self.img_dir, img_name)  # VALID_DATA_DIR와 이미지 파일 이름 결합
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        label = int(self.data_frame.iloc[idx, 1])
        return image, label, img_path  # img_path를 반환


val_dataset = CustomImageDataset(csv_file=valid_csv_file, img_dir=VALID_DATA_DIR, transform=val_transform)
valid_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=1, pin_memory=True)

# Validation 함수 정의
def validate_with_examples(model, valid_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    correct_examples = []
    incorrect_examples = []

    with torch.no_grad():
        for images, labels, img_paths in valid_loader:
            images, labels = images.to(device), labels.to(device)

            # 모델 예측
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            # 손실 계산
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # 예측 결과 저장
            total += 1
            if predicted == labels:
                correct += 1
                if len(correct_examples) < 4:
                    correct_examples.append((images.cpu(), predicted.item(), labels.item(), img_paths))
            else:
                if len(incorrect_examples) < 4:
                    incorrect_examples.append((images.cpu(), predicted.item(), labels.item(), img_paths))

    # 손실 및 정확도 계산
    val_loss = running_loss / total
    val_accuracy = 100 * correct / total

    return val_loss, val_accuracy, correct_examples, incorrect_examples

# Validation 수행 및 예제 추출
criterion = nn.CrossEntropyLoss()
_, val_accuracy, correct_examples, incorrect_examples = validate_with_examples(model, valid_loader, criterion, device)
print(f"Validation Accuracy: {val_accuracy:.2f}%")

# 한글 폰트 설정
plt.rc('font', family='NanumGothic')

# 알맞게 분류한 예제와 잘못 분류한 예제를 출력
def plot_examples(examples, title):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(title)
    for i, (img, pred, label, img_path) in enumerate(examples):
        img = img.squeeze().permute(1, 2, 0).numpy()
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f"Pred: {pred}, Label: {label}\n{os.path.basename(img_path[0])}")  # img_path[0]로 수정
    plt.show()

# 예제 시각화
plot_examples(correct_examples, "Correctly Classified Examples")
plot_examples(incorrect_examples, "Incorrectly Classified Examples")


# Mission 2-1

In [ ]:
# 각 파일명들을 분성하여 dataframe로 바꿔주는 함
def extract_features_from_directory(directory, extensions=['jpg', 'json']):
    data = {
        "File_Name": [],
        "W/T": [],
        "ImageID": [],
        "Era": [],
        "Style": [],
        "Gender": [],
        "R_id": []
    }
    
    # 디렉토리 내 파일 탐색
    for filename in os.listdir(directory):
        ext = filename.split(".")[-1]
        if ext in extensions:  # 지정한 확장자 파일만 처리
            data["File_Name"].append(filename)
            parts = filename.replace("." + ext, "").split("_")
            
            data["W/T"].append(parts[0])
            data["ImageID"].append(parts[1])
            data["Era"].append('20' + parts[2] if int(parts[2]) < 50 else '19' + parts[2])
            data["Style"].append(parts[3])
            data["Gender"].append("여성" if parts[4] == "W" else "남성")
            
            # json 파일일 경우 R_id 추가
            if ext == 'json':
                data["R_id"].append(parts[5])
            else:
                data["R_id"].append(None)
    
    return pd.DataFrame(data)

In [ ]:
train_label_dir = './data/training_label'
train_image_dir = './data/training_image'
valid_label_dir = './data/validation_label'
valid_image_dir = './data/validation_image'

#### Train dataset분석

In [ ]:
train_image_df = extract_features_from_directory(train_image_dir, 'jpg')
train_image_df.describe()

In [ ]:
# train_image_df에서 고유한 ImageID 추출 ('.jpg' 삭제)
train_file_name = train_image_df['ImageID'].str.replace('.jpg', '', regex=False).unique()
print(len(train_file_name))  # 고유한 ImageID 개수 출력

In [ ]:
train_label_df = extract_features_from_directory(train_label_dir, 'json')
train_label_df.describe()

In [ ]:
# train_file_name을 set으로 변환하여 비교
train_file_name_set = set(train_file_name)

# train_file_name_set에 일치하는 값만 필터링
filtered_train_label_df = train_label_df[
    train_label_df['ImageID'].str.replace('.json', '', regex=False).isin(train_file_name_set)
]

# 결과 확인
print(filtered_train_label_df.head())
print(f"필터링된 데이터 개수: {len(filtered_train_label_df)}")

In [ ]:
# "성별"과 "스타일"을 기준으로 그룹화한 후, 각 그룹의 "설문ID" 개수를 세기
gender_style_stats = filtered_train_label_df.groupby(['Gender', 'Style'])['R_id'].count().reset_index()

# 통계치가 담긴 데이터프레임에 컬럼명 변경 (옵션)
gender_style_stats.columns = ['Gender', 'Style', 'Survey_Count']

# Survey_Count의 총합 계산
total_survey_count = gender_style_stats['Survey_Count'].sum()

# '총합' 행 추가
total_row = pd.DataFrame({'Gender': ['총합'], 'Style': ['-'], 'Survey_Count': [total_survey_count]})
gender_style_stats = pd.concat([gender_style_stats, total_row], ignore_index=True)

# 결과 출력
print(gender_style_stats)

## Valid dataset 분석

In [ ]:
valid_image_df = extract_features_from_directory(valid_image_dir, 'jpg')
valid_image_df.describe()

In [ ]:
# valid_image_df에서 고유한 ImageID 추출
valid_image_id = valid_image_df['ImageID'].unique()
len(valid_image_id)

In [ ]:
valid_label_df = extract_features_from_directory(valid_label_dir, 'json')
valid_label_df.describe()

In [ ]:
# valid_image_id에 공통된 ImageID만 valid_label_df에서 필터링
filtered_valid_label_df = valid_label_df[valid_label_df['ImageID'].isin(valid_image_id)]

# 공통된 ImageID만 추출한 데이터프레임 출력
filtered_valid_label_df.describe()

In [ ]:
# "성별"과 "스타일"을 기준으로 그룹화한 후, 각 그룹의 "설문ID" 개수를 세기
gender_style_stats = filtered_valid_label_df.groupby(['Gender', 'Style'])['R_id'].count().reset_index()

# 통계치가 담긴 데이터프레임에 컬럼명 변경 (옵션)
gender_style_stats.columns = ['Gender', 'Style', 'Survey_Count']

# Survey_Count의 총합 계산
total_survey_count = gender_style_stats['Survey_Count'].sum()

# '총합' 행 추가
total_row = pd.DataFrame({'Gender': ['총합'], 'Style': ['-'], 'Survey_Count': [total_survey_count]})
gender_style_stats = pd.concat([gender_style_stats, total_row], ignore_index=True)

# 결과 출력
print(gender_style_stats)


# Mission 2-2

#### 2-1을 통해서 얻은 유효한 라벨링 data

In [ ]:
filtered_train_label_df.describe()

In [ ]:
filtered_valid_label_df.describe()

In [ ]:
# 스타일 선호 정보를 처리하는 함수
def process_style_preference(df, json_dir, data_type='train'):
    """
    df: DataFrame containing the Filename column.
    json_dir: Directory path where the json files are stored.
    data_type: 'train' 또는 'valid'로 구분하여 처리. (default: 'train')
    """
    
    # 새로운 데이터프레임을 위한 딕셔너리 생성
    style_preference_data = {
        "응답자 ID": [],
        f"{data_type.capitalize()} 스타일 선호": [],
        f"{data_type.capitalize()} 스타일 비선호": []
    }

    # JSON 파일들을 탐색하고 데이터 처리
    for idx, row in df.iterrows():
        filename = row['File_Name']  # json 파일명
        filepath = os.path.join(json_dir, filename)  # 파일 경로 설정

        # 파일 존재 여부 확인 후 처리
        if os.path.exists(filepath):
            # JSON 파일 읽기
            with open(filepath, 'r') as file:
                data = json.load(file)

            # 응답자 ID 및 Q5 값 추출
            respondent_id = data['user']['R_id']
            preference = data['item']['survey']['Q5']
            img_name = data['item']['imgName']

            # 응답자 ID 추가
            style_preference_data['응답자 ID'].append(respondent_id)

            # 선호 여부에 따라 분류
            if preference == 2:  # 선호
                style_preference_data[f"{data_type.capitalize()} 스타일 선호"].append(img_name)
                style_preference_data[f"{data_type.capitalize()} 스타일 비선호"].append(None)
            elif preference == 1:  # 비선호
                style_preference_data[f"{data_type.capitalize()} 스타일 선호"].append(None)
                style_preference_data[f"{data_type.capitalize()} 스타일 비선호"].append(img_name)
        else:
            print(f"File {filename} does not exist.")

    # 데이터프레임으로 변환
    df_style_preference = pd.DataFrame(style_preference_data)
    
    return df_style_preference

# Example usage:
# train 데이터 처리
train_result_df = process_style_preference(filtered_train_label_df, train_label_dir, data_type='train')

# valid 데이터 처리
valid_result_df = process_style_preference(filtered_valid_label_df, valid_label_dir, data_type='valid')

# 결과 출력
print(train_result_df)
print(valid_result_df)


In [ ]:
# 유효한 train data 분석시, 응답자 ID별 총 tuple개수
len(train_result_df['응답자 ID'].unique())

In [ ]:
# None 값을 제거하고 빈 리스트도 NaN으로 변환하는 함수
def remove_none_and_empty(lst):
    if isinstance(lst, list):
        # None을 제거한 리스트 반환, 빈 리스트일 경우 NaN으로 대체
        cleaned_list = [x for x in lst if x is not None]
        return cleaned_list if cleaned_list else np.nan  # 빈 리스트는 NaN으로 처리
    return lst

##### train dataset

In [ ]:
# 그룹화된 데이터에서 None을 제거 (열별로 apply 사용)
grouped_train = train_result_df.groupby('응답자 ID').agg(list).reset_index()

# 각 열에 대해 remove_none 함수 적용
for col in grouped_train.columns:
    if col != '응답자 ID':  # 응답자 ID 열은 처리하지 않음
        grouped_train[col] = grouped_train[col].apply(remove_none_and_empty)

# 결과 출력
print(grouped_train)

In [ ]:
# 유효한 valid data 분석시, 응답자 ID별 총 tuple개수
len(valid_result_df['응답자 ID'].unique())

##### valid dataset

In [ ]:
# 그룹화된 데이터에서 None을 제거 (열별로 apply 사용)
grouped_valid = valid_result_df.groupby('응답자 ID').agg(list).reset_index()

# 각 열에 대해 remove_none 함수 적용
for col in grouped_valid.columns:
    if col != '응답자 ID':  # 응답자 ID 열은 처리하지 않음
        grouped_valid[col] = grouped_valid[col].apply(remove_none_and_empty)

# 결과 출력
print(grouped_valid)

In [ ]:
# '응답자 ID'를 기준으로 inner join
merged_df = pd.merge(grouped_train, grouped_valid, on='응답자 ID', how='inner', suffixes=('_train', '_valid'))

# 각 행에 대해 리스트의 원소 개수를 합산하여 'Total_Count' 컬럼 추가
merged_df['Total_Count'] = merged_df.apply(lambda row: sum(len(val) if isinstance(val, list) else 0 for val in row[1:]), axis=1)

# 'Total_Count' 기준으로 내림차순 정렬
merged_df = merged_df.sort_values(by='Total_Count', ascending=False)

# 결과 출력
print(merged_df)

# 'total_count' 컬럼 삭제
merged_df = merged_df.drop(columns=['Total_Count'])

# Option. 전체 결과 저장 
# # CSV 파일로 저장
# file_path = './ALL_2_2.csv'
# merged_df.to_csv(file_path, index=False, encoding='utf-8-sig')
# print('경로 :', file_path, 'ALL 2-2 CSV 파일 저장완료')

In [ ]:
# 상위 100명 추출 후 인덱스 재설정
merged_df_100 = merged_df.head(100).reset_index(drop=True)

# 결과 출력
print(merged_df_100)

# CSV 파일로 저장
file_path = './2_2.csv'
merged_df_100.to_csv(file_path, index=False, encoding='utf-8-sig')
print('경로 :', file_path, '2-2 top 100 정답 CSV 파일 저장완료')

# Mission 3-1

 ![nn](./표1.jpg)

 ![nn](./포2.jpg)

#### 자세한 분석은 PPT에 첨부하였습니다. 참고 부탁드립니다.

# Mission 3-2

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [ ]:
set_seed()  # 시드 고정

In [ ]:
# 데이터셋 파라미터
TRAIN_DATA_DIR = './data/final_seg_train_image/'  # 학습 데이터 경로
VALID_DATA_DIR = './data/final_seg_valid_image'  # 검증 데이터 경로

In [ ]:
# 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"

# 모델 준비 및 가중치 로드
NUM_CLASSES = 31
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
checkpoint = torch.load("/home/user/Desktop/ICT/output/best_model.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

# 새로운 전처리 정의
val_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.2651, 0.2375, 0.2316], [0.2848, 0.2609, 0.2559])
])

In [ ]:
# 특성 추출 함수 정의
# 가장 마지막 layer에서 2번째, global avg pooling 결과를 latent vector로 사용
def get_feature_vector(img_path):
    img = val_transform(Image.open(img_path)).unsqueeze(0).to(device)
    with torch.no_grad():
        features = model.conv1(img)
        features = model.bn1(features)
        features = model.relu(features)
        features = model.maxpool(features)
        
        features = model.layer1(features)
        features = model.layer2(features)
        features = model.layer3(features)
        features = model.layer4(features)
        
        features = model.avgpool(features)
        feature_vector = features.view(features.size(0), -1)  # (1, 512) 형태로 변환
    return feature_vector.cpu().numpy().flatten()

In [ ]:
# 임베딩 생성 함수
def generate_embeddings(data_dir, embedding_output_path, index_output_path, desc="Generating Embeddings"):
    embeddings = []
    image_index = {}
    images = os.listdir(data_dir)

    # tqdm을 사용하여 진행 상태 표시
    for idx, img_name in enumerate(tqdm(images, desc=desc)):
        img_path = os.path.join(data_dir, img_name)
        embedding = get_feature_vector(img_path)
        embeddings.append(embedding)
        image_index[img_name] = idx  # 이미지 파일 이름과 인덱스 매핑

    # numpy 배열로 변환하여 하나의 임베딩 매트릭스 생성
    embeddings = np.vstack(embeddings)
    np.save(embedding_output_path, embeddings)
    print(f"임베딩 매트릭스가 {embedding_output_path}에 저장되었습니다.")

    # 이미지 파일 이름과 인덱스 매핑 사전을 .json 파일로 저장
    with open(index_output_path, "w", encoding="utf-8") as f:
        json.dump(image_index, f, ensure_ascii=False, indent=4)
    print(f"이미지 파일 이름과 인덱스 매핑이 {index_output_path}에 저장되었습니다.")

    # 임베딩 매트릭스와 인덱스 정보 확인
    print("임베딩 매트릭스 Shape:", embeddings.shape)
    print("예시 - 첫 번째 이미지의 파일 이름:", images[0])
    print("예시 - 첫 번째 이미지의 인덱스:", image_index[images[0]])

In [ ]:
# Train/valid 이미지 set 각각 임베딩 생성
generate_embeddings(TRAIN_DATA_DIR, "./train_embeddings.npy", "./train_image_index.json", desc="Train Embeddings")
generate_embeddings(VALID_DATA_DIR, "./valid_embeddings.npy", "./valid_image_index.json", desc="Validation Embeddings")

In [ ]:
# 데이터 준비
file_path = "./2_2.csv"  # 실제 경로로 변경
df = pd.read_csv(file_path)

# 임베딩 및 인덱스 매핑 파일 로드
train_embeddings = np.load("./train_embeddings.npy")
valid_embeddings = np.load("./valid_embeddings.npy")
with open("./train_image_index.json", "r", encoding="utf-8") as f:
    train_index = json.load(f)
with open("./valid_image_index.json", "r", encoding="utf-8") as f:
    valid_index = json.load(f)

# 결과 저장을 위한 리스트 및 총 valid 원소 개수 카운트 변수
all_results = []
total_valid_elements = 0  # 총 valid 원소 개수를 저장할 변수
# threshold = 0.5  # 임계값 설정

##### train/valid row 무결성 재확인

In [ ]:
# 동시에 빈 경우의 개수 확인
empty_count = df[(df['Train 스타일 선호'].isna()) & (df['Train 스타일 비선호'].isna())].shape[0]

print(f"'Train 스타일 선호'와 'Train 스타일 비선호'가 동시에 빈 경우의 개수: {empty_count}")

In [ ]:
# 'Valid 스타일 선호'와 'Valid 스타일 비선호'가 동시에 빈 경우의 개수 확인
empty_count = df[(df['Valid 스타일 선호'].isna()) & (df['Valid 스타일 비선호'].isna())].shape[0]

print(f"'Valid 스타일 선호'와 'Valid 스타일 비선호'가 동시에 빈 경우의 개수: {empty_count}")

#### 평가

In [ ]:
# 결과와 관련된 변수 초기화
all_results = []
total_valid_elements = 0

# 결과 데이터를 저장할 디렉토리
output_directory = "./data"
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# 데이터 프레임 순회
for idx, row in df.iterrows():
    # train 선호/비선호, valid 선호/비선호 이미지 파일명 리스트
    train_preferred = row['Train 스타일 선호'] if pd.notna(row['Train 스타일 선호']) else ""
    train_non_preferred = row['Train 스타일 비선호'] if pd.notna(row['Train 스타일 비선호']) else ""
    valid_preferred = row['Valid 스타일 선호'] if pd.notna(row['Valid 스타일 선호']) else ""
    valid_non_preferred = row['Valid 스타일 비선호'] if pd.notna(row['Valid 스타일 비선호']) else ""
    
    # 리스트 변환
    train_preferred = train_preferred.strip("[]").replace("'", "").split(", ") if train_preferred else []
    train_non_preferred = train_non_preferred.strip("[]").replace("'", "").split(", ") if train_non_preferred else []
    valid_preferred = valid_preferred.strip("[]").replace("'", "").split(", ") if valid_preferred else []
    valid_non_preferred = valid_non_preferred.strip("[]").replace("'", "").split(", ") if valid_non_preferred else []
    
    # 각 그룹의 임베딩 리스트 생성
    train_preferred_embeddings = np.array([train_embeddings[train_index[img]] for img in train_preferred if img in train_index])
    train_non_preferred_embeddings = np.array([train_embeddings[train_index[img]] for img in train_non_preferred if img in train_index])
    
    # Valid 이미지가 있는 경우에만 유사도 계산 및 예측 진행
    if valid_preferred or valid_non_preferred:
        row_results = []
        
        # valid 선호와 비선호의 모든 이미지 순회
        for img_name in valid_preferred + valid_non_preferred:
            if img_name in valid_index:  # 유효한 이미지인지 확인
                valid_embedding = valid_embeddings[valid_index[img_name]].reshape(1, -1)
                
                # 유사도 계산
                if train_preferred_embeddings.size > 0:
                    preferred_similarities = cosine_similarity(valid_embedding, train_preferred_embeddings).flatten()
                    top_preferred = np.sort(preferred_similarities)[-10:]
                else:
                    top_preferred = np.array([])

                if train_non_preferred_embeddings.size > 0:
                    non_preferred_similarities = cosine_similarity(valid_embedding, train_non_preferred_embeddings).flatten()
                    top_non_preferred = np.sort(non_preferred_similarities)[-10:]
                else:
                    top_non_preferred = np.array([])
                
                # 다수결로 선호/비선호 결정
                votes_preferred = len([sim for sim in top_preferred if sim > 0.99])  
                votes_non_preferred = len([sim for sim in top_non_preferred if sim > 0.99])
                
                if votes_preferred > votes_non_preferred:
                    prediction = "선호"
                elif votes_preferred < votes_non_preferred:
                    prediction = "비선호"
                else:
                    prediction = "선호" if votes_preferred >= 5 else "비선호"  #
                
                true_label = "선호" if img_name in valid_preferred else "비선호"
                
                # 결과 저장
                row_results.append({
                    "행 인덱스": idx,
                    "이미지 이름": img_name,
                    "예측": prediction,
                    "실제 라벨": true_label
                })
                
                # 총 valid 원소 개수 증가
                total_valid_elements += 1
            #else:
                #print('인덱스에 없는 이미지:', img_name)
        
        # 현재 행에 대한 결과 추가
        all_results.extend(row_results)

# 결과를 데이터프레임으로 저장
results_df = pd.DataFrame(all_results)
results_df.to_csv(f"{output_directory}/valid_predictions_by_row.csv", index=False)
print("개별 행에 대한 Validation 예측 결과가 ./data/valid_predictions_by_row.csv에 저장되었습니다.")

# 전체 정확도 계산
accuracy = (results_df["예측"] == results_df["실제 라벨"]).mean()
print(f"Validation 이미지 평가 정확도: {accuracy:.2%}")

# 총 valid 원소 개수 출력
print(f"총 처리한 valid 원소 개수: {total_valid_elements}")